In [1]:
import numpy as np
from cobra.util.array import create_stoichiometric_matrix
from docplex.mp.model import Model
from cobra.io import read_sbml_model

In [2]:
"""
Simple Test Implementation: MILP for Shortest MCS
Based on von Kamp & Klamt (2014) - Equations 1, 2, 7-11

This is a minimal working example to test the core MILP formulation.
"""

import numpy as np
from docplex.mp.model import Model

def build_dual_matrix(N, target):
    # Transpose S matrix
    D_S = N.transpose()
    # Dual identity matrix (vp's) and concatenate to D_S
    I = np.identity(len(D_S))
    D_S = np.concatenate([D_S, I], axis=1)
    # Dual identity matrix (vn's) and concatenate to D_S
    D_S = np.concatenate([D_S, I*-1], axis=1)
    # Secrete target metabolites (w)
    w = np.zeros(len(D_S), dtype="int")
    w[target] = -1
    # Concatenate D_S with w
    D_S = np.column_stack((D_S, w))
    return D_S

def enumerate_k_shortest_mcs(N, target_idx, irrev, reaction_names, N_dual = None, k=5, ):
    """
    Enumerate k shortest mcss using exclusion constraints (Equation 11).
    
    Args:
        N: Stoichiometric matrix
        irreversible_reactions: List of irreversible reaction indices
        k: Number of shortest mcss to find
        
    Returns:
        List of mcs dictionaries
    """
    m, n = N.shape
    if N_dual is None:
        N_dual = build_dual_matrix(N,target_idx)
    m_dual, n_dual = N_dual.shape
    mcss = []
    
    # Create base model
    mdl = Model(name="k_Shortest_mcss")
    
    # Variables for dual system (equation 17)
    # u (metabolite duals), vp/vn (reaction duals split), w (target constraint dual)
    u = mdl.continuous_var_list(m, lb=-mdl.infinity, name='u')
    vp = mdl.continuous_var_list(n, lb=0, name='vp')
    vn = mdl.continuous_var_list(n, lb=0, name='vn')
    w = mdl.continuous_var_list(1, lb=0, name='w')
    
    # Combine into single variable vector for dual system
    dual_vars = u + vp + vn + w

    # Indicator variables
    zp = mdl.binary_var_list(n, name='z_p')
    zn = mdl.binary_var_list(n, name='z_n')
    
    # Dual steady state constraints (equation 17)
    for i in range(m_dual):

        if irrev[i]:
            mdl.add_constraint(
                mdl.sum(N_dual[i, j] * dual_vars[j] for j in range(n_dual)) >= 0,
                ctname=f'dual_geq_{i}'
            )
        else:
            mdl.add_constraint(
                mdl.sum(N_dual[i, j] * dual_vars[j] for j in range(n_dual)) == 0,
                ctname=f'dual_ss_{i}'
            )
        
    # Additional constraint: b^T w <= -c (equation 17)
    # With b=1 and c=1: w >= 1
    mdl.add_constraint(w[0] >= 1, ctname='target_constraint')
    
    # Indicator constraints for vp and vn (equation 6 adapted)
    for i in range(n):
        # zp[i] = 0 <=> vp[i] = 0
        mdl.add_indicator(zp[i], vp[i] == 0, active_value=0, name=f'ind_vp_zero_{i}')
        # zp[i] = 1 <=> vp[i] >= 1
        mdl.add_indicator(zp[i], vp[i] >= 1, active_value=1, name=f'ind_vp_active_{i}')
        
        # zn[i] = 0 <=> vn[i] = 0
        mdl.add_indicator(zn[i], vn[i] == 0, active_value=0, name=f'ind_vn_zero_{i}')
        # zn[i] = 1 <=> vn[i] >= 1
        mdl.add_indicator(zn[i], vn[i] >= 1, active_value=1, name=f'ind_vn_active_{i}')
        
        # Constraint (18): zp[i] + zn[i] <= 1
        mdl.add_constraint(zp[i] + zn[i] <= 1, ctname=f'single_direction_{i}')

    
    # At least one reaction in MCS (equation 9 adapted)
    mdl.add_constraint(
        mdl.sum(zp[i] + zn[i] for i in range(n)) >= 1,
        ctname='at_least_one'
    )
    
    # Objective: minimize MCS size (equation 19)
    mdl.minimize(mdl.sum(zp[i] + zn[i] for i in range(n)))
    
    # Iteratively find k shortest mcss
    for iteration in range(k):
        solution = mdl.solve(log_output=False)
        
        if solution is None:
            print(f"No more mcss found after {iteration} iterations")
            break
        
        # Extract solution
        zp_values = [int(zp[i].solution_value) for i in range(n)]
        zn_values = [int(zn[i].solution_value) for i in range(n)]
        z_combined = [zp_values[i] + zn_values[i] for i in range(n)]
        
        dual_vars_values = [dual_vars[i].solution_value for i in range(m+n+n+1)]

        # Map active indices → reaction names
        mcs_reaction_names = [reaction_names[i] for i, val in enumerate(z_combined) if val == 1]
        mcs_size = sum(z_combined)

        
        mcs = {
            'fluxes': dual_vars_values,
            'size': mcs_size,
            'reactions': mcs_reaction_names
        }
        mcss.append(mcs)
        
        # Add exclusion constraint - Equation (11)
        # sum(z_tilde[i] * z[i]) <= sum(z_tilde[i]) - 1
        mdl.add_constraint(
            mdl.sum(z_combined[i] * (zp[i] + zn[i]) for i in range(n))
            <= sum(z_combined) - 1,
            ctname=f'exclusion_{iteration}'
        )
#        )
        
        print(f"MCS {iteration+1}: size={mcs['size']}, reactions={mcs['reactions']}")
    
    return mcss



In [3]:
model_name = '../../models/M_model'
model = read_sbml_model(model_name + ".xml")
S = create_stoichiometric_matrix(model)
target_rxn = "r5"
target = model.reactions.index(target_rxn)
rxn_names = [rxn.id for rxn in model.reactions]

irrev = [not rxn.reversibility for rxn in model.reactions ]
mcs = enumerate_k_shortest_mcs(S, target, irrev, rxn_names, k=20)

MCS 1: size=1, reactions=['r5']
MCS 2: size=2, reactions=['r1', 'r2']
MCS 3: size=2, reactions=['r3', 'r4']
No more mcss found after 3 iterations


In [4]:
model_name = '../../models/PQS_model'
model = read_sbml_model(model_name + ".xml")
S = create_stoichiometric_matrix(model)
target_rxn = "R04"
target = model.reactions.index(target_rxn)
rxn_names = [rxn.id for rxn in model.reactions]

irrev = [not rxn.reversibility for rxn in model.reactions ]
mcs = enumerate_k_shortest_mcs(S, target, irrev, rxn_names, k=20)

MCS 1: size=1, reactions=['R01']
MCS 2: size=1, reactions=['R04']
MCS 3: size=2, reactions=['R10', 'R11']
MCS 4: size=2, reactions=['R05', 'R10']
MCS 5: size=2, reactions=['R05', 'R06']
MCS 6: size=2, reactions=['R03', 'R05']
MCS 7: size=3, reactions=['R03', 'R07', 'R11']
MCS 8: size=3, reactions=['R06', 'R07', 'R11']
No more mcss found after 8 iterations


In [5]:
model_name = '../../models/ecoli5010_no_b'
model = read_sbml_model(model_name + ".xml")
S = create_stoichiometric_matrix(model)
target_rxn = "Biomass_Ecoli_core_w_GAM"
target = model.reactions.index(target_rxn)
rxn_names = [rxn.id for rxn in model.reactions]

irrev = [not rxn.reversibility for rxn in model.reactions]
mcs = enumerate_k_shortest_mcs(S, target, irrev, rxn_names, k=500)

MCS 1: size=1, reactions=['GAPD']
MCS 2: size=1, reactions=['PGK']
MCS 3: size=1, reactions=['RPI']
MCS 4: size=1, reactions=['ICDHyr']
MCS 5: size=1, reactions=['Biomass_Ecoli_core_w_GAM']
MCS 6: size=1, reactions=['NH4t']
MCS 7: size=1, reactions=['EX_nh4_e']
MCS 8: size=1, reactions=['PGM']
MCS 9: size=1, reactions=['ENO']
MCS 10: size=1, reactions=['CS']
MCS 11: size=1, reactions=['GLCpts']
MCS 12: size=1, reactions=['EX_glc_e']
MCS 13: size=1, reactions=['ACONTa']
MCS 14: size=1, reactions=['ACONTb']
MCS 15: size=2, reactions=['PGI', 'G6PDH2r']
MCS 16: size=2, reactions=['PFK', 'G6PDH2r']
MCS 17: size=2, reactions=['G6PDH2r', 'RPE']
MCS 18: size=2, reactions=['FBA', 'G6PDH2r']
MCS 19: size=2, reactions=['RPE', 'TKT1']
MCS 20: size=2, reactions=['PGI', 'NADTRHD']
MCS 21: size=2, reactions=['TPI', 'G6PDH2r']
MCS 22: size=2, reactions=['MDH', 'PPC']
MCS 23: size=2, reactions=['RPE', 'TALA']
MCS 24: size=2, reactions=['FBA', 'EDP1']
MCS 25: size=2, reactions=['GND', 'RPE']
MCS 26: siz